# enrich_run — LÀM GIÀU ĐOẠN: ghép TIÊU ĐỀ vào đoạn trước khi chấm (việc 2)

**Vì sao chạy cái này:** 23/08, phép đối chứng `ce_ours` trong `hard15_run` cho thấy chính
`AITeamVN/Vietnamese_Reranker` bắt được **6/15** câu khó khi được xem thêm 260 ký tự đầu
văn bản, so với **3/15** khi chỉ xem đoạn (cùng model, cùng rổ, cùng luật chốt n=0, và
6 câu ⊃ 3 câu — không đánh đổi câu nào). Tức **tiêu đề là thứ bộ chấm đang thiếu.**

**⚠️ NHƯNG ĐÃ CÓ MỘT LẦN THUA, ĐỌC TRƯỚC:** 12/08 ghép `name` của D — **slug URL KHÔNG
DẤU**, ghép vào ĐẦU đoạn — làm **tệ đi 2,0 điểm**. CLAUDE.md đã chẩn ba cơ chế: (1) mất
dấu nên không khớp câu hỏi, (2) chiếm mất vị trí đầu vốn được cross-encoder nặng trọng số,
(3) chuỗi ASCII nối gạch ngang lệch phân phối huấn luyện. Bản cũ **chưa từng thử vị trí
CUỐI**, nên chưa tách được cơ chế 1 khỏi cơ chế 2.

Lượt này tách hẳn ra: tiêu đề lấy từ **chính `passage`** (có dấu, đúng phân phối), và
quét **cả hai vị trí**.

**⚠️ 15 câu khó KHÔNG chứng minh được gì về thiệt hại.** Chúng được chọn vì bài mình sai,
nên chỉ đo được phần ĂN THÊM. Lần thua 12/08 thua chính vì **phá các câu đang đúng**.
Đó là lý do lượt này chạy TRỌN dev300 chứ không phải 15 câu.

### Ba nhánh, cùng đoạn trích, chỉ khác chỗ đặt tiêu đề

| tag | văn bản đưa cho model |
|---|---|
| `plain` | `excerpt` — **ĐỐI CHỨNG**, không tiêu đề |
| `front` | `head` + `excerpt` — **đúng cấu hình đã đo 6/15** |
| `back` | `excerpt` + `head` — trả lời câu hỏi treo từ 13/08 |

Ba nhánh dùng CHUNG một `excerpt` do `pick_chunks` chọn → khác biệt duy nhất là tiêu đề.
So `front`/`back` với `plain`, **đừng** so với 0.9350 (đó là cấu hình hai tầng khác hẳn).

### Ngưỡng đặt TRƯỚC khi chạy — so với nhánh `plain`, ở `n` tốt nhất của mỗi nhánh

| Δ so với `plain` | quyết định |
|---|---|
| **≥ +2,0** | thắng rõ → nhét vào `deep_chunk.pick_chunks`, chạy lượt hai tầng ~3h |
| +0,7 … +2,0 | **vùng mù** (dev300 nhiễu ±1,5 câu ≈ ±0,5 điểm). KHÔNG mang lên đề thi |
| −0,7 … +0,7 | hoà → đóng, tiêu đề không phải cần gạt |
| **≤ −0,7** | thua như 12/08 dù đã có dấu → **đóng vĩnh viễn hướng ghép tiêu đề** |

**Chi phí:** 300 câu × 50 ứng viên × 3 nhánh = **45.000 cặp**. AITeamVN đo thật 12,9 cặp/s
→ **~58 phút GPU**. Rẻ hơn lượt hai tầng (~3h) đúng ba lần, mà tách được biến sạch hơn.

**Upload lên dataset `project-ir`:** `deep_chunk.py` · `make_candidates_fallback.py` ·
`rerank.py` · `rerank_qwen.py` · `fusion_rrf_top50_dev_1000.json` · `dev_300_locked.json`
· `selected-contexts`. Tất cả đã có sẵn từ `fusion_run` — **không cần upload gì mới.**

> 🔴 Nhớ bật GPU (Settings → Accelerator). Bước 1 có assert chặn.


In [ ]:
!pip install -q sentence-transformers

In [ ]:
# ===== Bước 1: cấu hình, dựng 3 nhánh văn bản trên CPU =====
import os, sys, json, time, hashlib, gc

import torch
assert torch.cuda.is_available(), "KHÔNG CÓ GPU. Settings -> Accelerator -> GPU rồi chạy lại."
print(f"GPU: {torch.cuda.get_device_name(0)}")

HEAD_CHARS, EXC_CHARS, SEP = 260, 900, "\n\n"   # y hệt hard15_run -> so sánh được
TOPK = 5

INPUT_DIR = next(p for p in ("/kaggle/input/project-ir",
                             "/kaggle/input/datasets/locdovan211/project-ir")
                 if os.path.isdir(p))
CTX_DIR = next(p for p in (f"{INPUT_DIR}/selected-contexts/selected-contexts",
                           f"{INPUT_DIR}/selected-contexts")
               if os.path.isdir(p) and any(f.startswith("context_") for f in os.listdir(p)))
OUT = "/kaggle/working/outputs"; os.makedirs(OUT, exist_ok=True)
sys.path.append(INPUT_DIR)

for f in ("deep_chunk.py", "rerank.py"):
    b = open(f"{INPUT_DIR}/{f}", "rb").read()
    print(f"{f:28} {len(b):>6} bytes  {hashlib.sha256(b).hexdigest()[:12]}")
import deep_chunk as DC
from rerank import load_reranker
DC.MERGE_CHARS = 1800                       # cùng cấu hình bài chốt

cand = json.load(open(f"{INPUT_DIR}/fusion_rrf_top50_dev_1000.json", encoding="utf-8"))
dev  = json.load(open(f"{INPUT_DIR}/dev_300_locked.json", encoding="utf-8"))
qids = [q for q in dev if q in cand]
assert len(qids) == len(dev), f"THIẾU {len(dev)-len(qids)} câu trong rổ fusion"
GOLD  = {q: {str(a) for a in dev[q]["answer"]} for q in qids}
ORDER = {q: [str(c["doc_id"]) for c in
             sorted(cand[q], key=lambda c: -float(c["rrf_score"]))] for q in qids}

t0 = time.time()
index, head, exc = [], [], []               # index[i] = (qid, doc_id)
for i, q in enumerate(qids, 1):
    qs = dev[q]["question"]
    for c in cand[q]:
        d = str(c["doc_id"])
        e = DC.pick_chunks(qs, CTX_DIR, d, k=1)
        index.append((q, d))
        head.append(DC.read_passage(CTX_DIR, d)[:HEAD_CHARS].strip())
        exc.append((e[0] if e else "")[:EXC_CHARS])
    if i % 50 == 0:
        print(f"  băm {i}/{len(qids)} câu | {(time.time()-t0)/60:.1f} phút", flush=True)

ARMS = {"plain": lambda h, e: e,
        "front": lambda h, e: (h + SEP + e) if h else e,
        "back":  lambda h, e: (e + SEP + h) if h else e}
QTEXT = [dev[q]["question"] for q, _ in index]

print(f"\n{len(qids)} câu · {len(index)} cặp/nhánh · {len(ARMS)} nhánh"
      f" = {len(index)*len(ARMS):,} cặp")
print(f"ước {len(index)*len(ARMS)/12.9/60:.0f} phút GPU @12,9 cặp/s")
print(f"tiêu đề rỗng: {sum(1 for h in head if not h)} · đoạn rỗng: {sum(1 for e in exc if not e)}")
print(f"TRẦN rổ 50: {sum(len(GOLD[q] & set(ORDER[q]))/len(GOLD[q]) for q in qids)/len(qids):.4f}"
      f"   (CLAUDE.md ghi 0.9817 — lệch là sai chỗ nạp rổ, DỪNG)")

In [ ]:
# ===== Bước 2: chấm 3 nhánh, lưu scores sau MỖI nhánh =====
score_fn = load_reranker("AITeamVN/Vietnamese_Reranker", device="cuda", max_length=1024)

SC = {}
for tag, f in ARMS.items():
    p = f"{OUT}/scores_dev300_enrich_{tag}.json"
    if os.path.isfile(p):
        SC[tag] = json.load(open(p, encoding="utf-8")); print(f"{tag}: nạp lại"); continue
    t0 = time.time()
    s = score_fn.predict([[q, f(h, e)] for q, h, e in zip(QTEXT, head, exc)])
    per = {}
    for (q, d), v in zip(index, s):
        per.setdefault(q, {})[d] = {"ce": float(v), "bm25": 0.0}
    for q in per:                                   # rrf vào khoá bm25, y như fusion_run
        for r, d in enumerate(ORDER[q]):
            if d in per[q]: per[q][d]["bm25"] = -float(r)
    SC[tag] = per
    json.dump(per, open(p, "w", encoding="utf-8"), ensure_ascii=False)
    print(f"{tag}: xong {(time.time()-t0)/60:.1f} phút -> {p}", flush=True)
    gc.collect(); torch.cuda.empty_cache()
print(f"\nĐÃ LƯU {OUT}/ — TẢI VỀ TRƯỚC KHI ĐÓNG PHIÊN (quy tắc 2)")

In [ ]:
# ===== Bước 3: đo, đọc theo ngưỡng đã đặt trước =====
from rerank_from_d import blend_bm25_first

def rec(per, n):
    tot = 0.0
    for q in qids:
        r = sorted(per[q], key=lambda d: -per[q][d]["ce"])
        p = blend_bm25_first(r, ORDER[q], k=TOPK, n_bm25=n)
        tot += len(GOLD[q] & set(p)) / len(GOLD[q])
    return tot / len(qids)

NS = (0, 1, 2, 3)
tab = {t: {n: rec(SC[t], n) for n in NS} for t in ARMS if t in SC}
print(f"{'nhánh':8s}" + "".join(f"   n={n}  " for n in NS) + "   tốt nhất")
for t, r in tab.items():
    b = max(r.values())
    print(f"{t:8s}" + "".join(f"  {r[n]:.4f}" for n in NS) + f"   {b:.4f}")

base = max(tab["plain"].values())
print("\n" + "=" * 62)
for t in ("front", "back"):
    if t not in tab: continue
    d = (max(tab[t].values()) - base) * 100
    v = ("THẮNG RÕ -> nhét vào deep_chunk.pick_chunks, chạy lượt 2 tầng" if d >= 2.0 else
         "VÙNG MÙ -> KHÔNG mang lên đề thi"                             if d >= 0.7 else
         "HOÀ -> đóng, tiêu đề không phải cần gạt"                      if d > -0.7 else
         "THUA -> đóng VĨNH VIỄN hướng ghép tiêu đề (xác nhận 12/08)")
    print(f"{t:6s} Δ so plain = {d:+.2f} điểm   {v}")
print("=" * 62)
print("Nhắc: đây là cấu hình MỘT đoạn/văn bản, KHÔNG so được với 0.9350 (hai tầng).")